In [27]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime, timedelta


import pandas as pd


In [ ]:
# pip list
# pip install pandas

In [17]:
spark = (
    SparkSession
    .builder
    .appName("Increment")
    .getOrCreate()
)

In [2]:
# load data
df = pd.read_csv('apple_stock_data.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# create "history" - first snapshot make in 2010-01-01
snapshot_date = '2010-01-01'
historical = df[df['Date'] < snapshot_date].copy()
historical['ctl_loaded_at'] = datetime.now()

# new data (for last day's)
new_data = df[df['Date'] >= snapshot_date].copy()

print(f"Historical data: {historical}")
print(f"New writes: {len(new_data)}")

Historical data:            Date    Open    High     Low   Close    Volume  Adj Close  \
0    1984-09-07   26.50   26.87   26.25   26.50   2981600       3.02   
1    1984-09-10   26.50   26.62   25.87   26.37   2346400       3.01   
2    1984-09-11   26.62   27.37   26.62   26.87   5444000       3.07   
3    1984-09-12   26.87   27.00   26.12   26.12   4773600       2.98   
4    1984-09-13   27.50   27.62   27.50   27.50   7429600       3.14   
...         ...     ...     ...     ...     ...       ...        ...   
6382 2009-12-24  203.55  209.35  203.35  209.04  17888900     209.04   
6383 2009-12-28  211.72  213.95  209.61  211.61  23020200     211.61   
6384 2009-12-29  212.63  212.72  208.73  209.10  15900200     209.10   
6385 2009-12-30  208.83  212.00  208.31  211.64  14717300     211.64   
6386 2009-12-31  213.13  213.35  210.56  210.73  12586100     210.73   

                  ctl_loaded_at  
0    2026-03-25 21:16:32.550931  
1    2026-03-25 21:16:32.550931  
2    2026-03-25 

In [3]:
def simulate_incremental_load(historical_df, incoming_df, key_column=['Date']):
    """Simulation incremental"""

    # full outer join
    merged = incoming_df.merge(
        historical_df,
        on=key_column,
        how='outer',
        suffixes=('_new', '_old'),
        indicator=True
    )

    # determine action
    def get_action(row):
        if row['_merge'] == 'left_only':
            return 'I'
        elif row['_merge'] == 'right_only':
            return 'D'
        else:
            for col in ['Open','High','Low','Volume']:
                if row[f'{col}_new'] != row[f"{col}_old"]:
                    return 'U'
            return None

    merged['ctl_action'] = merged.apply(get_action, axis=1)

    # filtered only change (not Null)
    changes = merged[merged['ctl_action'].notna()].copy()

    # final columns
    for col in ['Open','High','Low', 'Close','Volume']:
        changes[col] = changes[f'{col}_new'].fillna(changes[f'{col}_old'])

    changes['Date'] = changes['Date']

    return changes[['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'ctl_action']]

In [4]:
# День 1: загружаем данные за 2010-01-04
day1_data = df[df['Date'] == '2010-01-04'].copy()
changes = simulate_incremental_load(historical, day1_data)
print("День 1 изменения:")
print(changes)

День 1 изменения:
           Date    Open    High     Low   Close      Volume ctl_action
0    1984-09-07   26.50   26.87   26.25   26.50   2981600.0          D
1    1984-09-10   26.50   26.62   25.87   26.37   2346400.0          D
2    1984-09-11   26.62   27.37   26.62   26.87   5444000.0          D
3    1984-09-12   26.87   27.00   26.12   26.12   4773600.0          D
4    1984-09-13   27.50   27.62   27.50   27.50   7429600.0          D
...         ...     ...     ...     ...     ...         ...        ...
6383 2009-12-28  211.72  213.95  209.61  211.61  23020200.0          D
6384 2009-12-29  212.63  212.72  208.73  209.10  15900200.0          D
6385 2009-12-30  208.83  212.00  208.31  211.64  14717300.0          D
6386 2009-12-31  213.13  213.35  210.56  210.73  12586100.0          D
6387 2010-01-04  213.43  214.50  212.38  214.01  17633200.0          I

[6388 rows x 7 columns]


In [5]:
# День 2: Загружаем за данные за 2010-01-05
day2_data = df[df['Date'] == '2010-01-05'].copy()
changes2 = simulate_incremental_load(historical, day2_data)
print('\nДень 2 изменения:')
print(changes2)


День 2 изменения:
           Date    Open    High     Low   Close      Volume ctl_action
0    1984-09-07   26.50   26.87   26.25   26.50   2981600.0          D
1    1984-09-10   26.50   26.62   25.87   26.37   2346400.0          D
2    1984-09-11   26.62   27.37   26.62   26.87   5444000.0          D
3    1984-09-12   26.87   27.00   26.12   26.12   4773600.0          D
4    1984-09-13   27.50   27.62   27.50   27.50   7429600.0          D
...         ...     ...     ...     ...     ...         ...        ...
6383 2009-12-28  211.72  213.95  209.61  211.61  23020200.0          D
6384 2009-12-29  212.63  212.72  208.73  209.10  15900200.0          D
6385 2009-12-30  208.83  212.00  208.31  211.64  14717300.0          D
6386 2009-12-31  213.13  213.35  210.56  210.73  12586100.0          D
6387 2010-01-05  214.60  215.59  213.25  214.38  21496600.0          I

[6388 rows x 7 columns]


In [19]:
d = spark.createDataFrame(changes2)

In [22]:
data = (
    d.filter(
        F.col('ctl_action') == 'D'
    )
)

In [24]:
d.printSchema()

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- ctl_action: string (nullable = true)



In [26]:
data.count()

Py4JJavaError: An error occurred while calling o96.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 2 in stage 3.0 failed 1 times, most recent failure: Lost task 2.0 in stage 3.0 (TID 14) (Kubik executor driver): java.io.IOException: Cannot run program "python3": CreateProcess error=3, Системе не удается найти указанный путь
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1143)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1073)
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:247)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:158)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:309)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:72)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.ap